# 03 — CW Recoverability and Noise-Transition Study

This notebook frames the transition question:

- when do multiple CWs provide enough independent pulsar-term phase constraints to recover distances?
- when do intrinsic red noise and the HD-correlated stochastic GWB erase those constraints?
- how does this depend on CW strain, number of CWs, GWB amplitude, and pairwise optimizer settings?

Main path uses `DATA_MODE="stochastic"`: the same enterprise simulation pattern as `data_likelihood_sandbox.ipynb` injects white noise, intrinsic red noise, HD GWB, and CWs; discovery evaluates a matching GP likelihood while distance-only optimization proceeds.


In [ ]:
from pathlib import Path
import itertools, subprocess, shlex, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
DRIVER = HERE / "02_multicw_pairwise_coordinate_ascent.py"
OUTDIR = HERE / "02_multicw_pairwise_outputs"
OUTDIR.mkdir(exist_ok=True)


## Transition Sweep Design

We scan CW strength and `N_CW`, then vary `gwb_log10_A`. Intrinsic red-noise amplitudes/gammas are drawn by `generate_injection_params`; GWB hyperparameters are fixed to the grid values and held fixed in the likelihood. That isolates distance-recovery behavior from noise-hyperparameter fitting.


In [ ]:
N_PSR = 20
N_CHAINS = 3
MAX_SWEEPS = 2
PAIR_MODE = "anchor_ladder"
MAX_PAIRS_PER_SWEEP = 15
MIN_POINTS = 21
POINTS_PER_MODE = 4
PROGRESS_EVERY = 10
COMPONENTS = 10
STOCHASTIC_SCENARIO = "well_separated"

N_CW_GRID = [2, 3, 4]
LOG10_H_GRID = [-13.0, -12.5, -12.0]
GWB_LOG10_A_GRID = [-18.5, -18.0, -17.5, -17.0, -16.5]
GWB_GAMMA = 13/3
RUN_TRANSITION_SWEEP = False


In [ ]:
def cmd_for(n_cw, log10_h, gwb_log10_a, idx):
    return [
        "rtk", "python", str(DRIVER.name),
        "--data-mode", "stochastic",
        "--stochastic-scenario", STOCHASTIC_SCENARIO,
        "--components", str(COMPONENTS),
        "--gwb-log10-a", str(gwb_log10_a),
        "--gwb-gamma", str(GWB_GAMMA),
        "--n-psr", str(N_PSR),
        "--n-cw", str(n_cw),
        "--n-chains", str(N_CHAINS),
        "--max-sweeps", str(MAX_SWEEPS),
        "--pair-mode", PAIR_MODE,
        "--max-pairs-per-sweep", str(MAX_PAIRS_PER_SWEEP),
        "--min-points", str(MIN_POINTS),
        "--points-per-mode", str(POINTS_PER_MODE),
        "--progress-every", str(PROGRESS_EVERY),
        "--log10-h", str(log10_h),
        "--seed", str(30000 + idx),
        "--noise-seed", str(40000 + idx),
    ]

jobs = []
for idx, (n_cw, logh, gwba) in enumerate(itertools.product(N_CW_GRID, LOG10_H_GRID, GWB_LOG10_A_GRID)):
    jobs.append((n_cw, logh, gwba, cmd_for(n_cw, logh, gwba, idx)))

for item in jobs[:10]:
    n_cw, logh, gwba, cmd = item
    print(f"n_cw={n_cw} log10_h={logh} gwb_log10_A={gwba}")
    print(shlex.join(cmd))
print(f"total jobs: {len(jobs)}")


In [ ]:
if RUN_TRANSITION_SWEEP:
    for idx, (n_cw, logh, gwba, cmd) in enumerate(jobs, 1):
        print("="*100)
        print(f"job {idx}/{len(jobs)} n_cw={n_cw} log10_h={logh} gwb_log10_A={gwba}")
        proc = subprocess.run(cmd, cwd=HERE, text=True, capture_output=True)
        print(proc.stdout[-5000:])
        if proc.returncode != 0:
            print(proc.stderr[-5000:])
            raise RuntimeError(f"job failed: n_cw={n_cw} logh={logh} gwba={gwba}")
else:
    print("RUN_TRANSITION_SWEEP=False; no jobs launched.")


## Analyze Transition Outputs


In [ ]:
files = sorted(OUTDIR.glob("run_*.json"))
rows = []
for path in files:
    data = json.loads(path.read_text())
    cfg = data.get("config", {})
    for res in data.get("results", []):
        rows.append({
            "file": path.name,
            "chain_id": res["chain_id"],
            "data_mode": cfg.get("data_mode", "pure"),
            "n_cw": cfg.get("n_cw"),
            "log10_h": cfg.get("log10_h"),
            "gwb_log10_a": cfg.get("gwb_log10_a"),
            "gwb_gamma": cfg.get("gwb_gamma"),
            "components": cfg.get("components"),
            "white": cfg.get("extra_white_rms", 0.0),
            "red": cfg.get("extra_red_rms", 0.0),
            "common_red": cfg.get("extra_common_red_rms", 0.0),
            "frac_half": res["final_score"]["frac_within_half_mode"],
            "median_modes": res["final_score"]["median_abs_modes"],
            "delta_lnL_truth": res["final_lnL"] - data["truth_lnL"],
        })
df = pd.DataFrame(rows)
if len(df):
    df = df[np.isfinite(df["frac_half"]) & np.isfinite(df["delta_lnL_truth"])]
print(df.shape)
df.tail()


In [ ]:
if len(df):
    df = df[df["data_mode"].fillna("pure") == "stochastic"].copy()
    agg = df.groupby(["n_cw", "log10_h", "gwb_log10_a"]).agg(
        frac_half=("frac_half", "mean"),
        best_frac_half=("frac_half", "max"),
        median_modes=("median_modes", "median"),
        best_delta_lnL=("delta_lnL_truth", "max"),
    ).reset_index()
    display(agg.tail(20))

    for n_cw, sub0 in agg.groupby("n_cw"):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for logh, sub in sub0.groupby("log10_h"):
            sub = sub.sort_values("gwb_log10_a")
            ax.plot(sub["gwb_log10_a"], sub["frac_half"], "o-", label=f"logh={logh}")
        ax.set_xlabel("GWB log10_A")
        ax.set_ylabel("mean fraction within 0.5 mode")
        ax.set_title(f"N_CW={n_cw}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
else:
    print("No outputs to analyze yet.")


## What To Look For

- Recovery improves with `N_CW` at fixed CW strain and GWB amplitude.
- Recovery drops as `gwb_log10_A` rises toward the CW residual scale.
- Stronger CWs should tolerate larger stochastic backgrounds before distance modes become ambiguous.
- Cases where final lnL exceeds truth under noisy injections are possible: one noise realization need not maximize exactly at injected distances.
- Current stochastic runs fix noise hyperparameters at injection truth. Next realism step: include noise-hyperparameter optimization/marginalization around this distance optimizer.
